## Prompt

> In the Enamine REAL database, find molecules with the lowest Tanimoto similarity and the highest possible ESP-cosine similarity to the query at the same time. List them sorted by ESP-cosine similarity, highest to lowest.

The public `deepmedchem` API only supports one similarity objective per query (confirmed live across four databases — see `note.md`), and a plain search response carries no secondary similarity score. So Tanimoto for the ESP-retrieved candidates cannot come from the API itself — the chatbot's runtime is confirmed to also have RDKit and pandas available, so it's computed locally with RDKit on the SMILES `deepmedchem` already returned. `deepmedchem` remains the sole source of retrieval/pricing/database data; RDKit only computes a fingerprint similarity between two SMILES strings already in hand, nothing more.

Approach: fetch the top 200 by ESP cosine (the one well-defined retrieval criterion — see `guidelines.md`), compute each hit's ECFP4/Morgan Tanimoto similarity to the query locally with RDKit, shortlist by the `esp / tanimoto` ratio (high ESP relative to how Tanimoto-different it is — the scaffold-hopping signal), then display that shortlist sorted by ESP-cosine similarity, highest to lowest.

Caveat: RDKit's `rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)` is the standard open reproduction of ECFP4, but its exact bit parameters are not guaranteed to match the server's own `rdkit.ecfp4_tanimoto` metric implementation bit-for-bit.

In [1]:
from deepmedchem import Client
from rdkit import Chem
from rdkit.Chem import DataStructs, rdFingerprintGenerator
import pandas as pd

DATABASE = "enamine-real-v5a"
QUERY = "CC(=O)Oc1ccccc1C(=O)O"  # aspirin
LIMIT = 200
TOP_N = 10

MORGAN_GENERATOR = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)


def ecfp4(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    return MORGAN_GENERATOR.GetFingerprint(mol)


with Client() as dmc:
    result = dmc.search(QUERY, database=DATABASE, method="esp", limit=LIMIT)

print(f"{len(result.hits)} hits, database={result.database_id} release={result.database_release}")

query_fp = ecfp4(QUERY)

rows = []
for hit in result.hits:
    tanimoto = DataStructs.TanimotoSimilarity(query_fp, ecfp4(hit.smiles))
    ratio = hit.score / tanimoto if tanimoto > 0 else float("inf")
    rows.append(
        {
            "smiles": hit.smiles,
            "esp": hit.score,
            "tanimoto": tanimoto,
            "esp_over_tanimoto": ratio,
            "price": hit.price,
        }
    )

df = pd.DataFrame(rows)
top = df.sort_values("esp_over_tanimoto", ascending=False).head(TOP_N)
top = top.sort_values("esp", ascending=False).reset_index(drop=True)
top

200 hits, database=enamine-real-v5a release=2026-09-06.1


,smiles,esp,tanimoto,esp_over_tanimoto,price
0,CC(C)(C)OC(=O)Oc1ccccc1C(=O)O,0.899872,0.606061,1.484788,163
1,COC(=O)Oc1ccccc1C(C)=O,0.864030,0.515152,1.677235,163
2,C=C(C)C(=O)Nc1ccccc1C(=O)O,0.765666,0.405405,1.888643,163
3,C=C(C)COc1ccccc1C(=O)Cc1ccccc1C(=O)O,0.738384,0.418605,1.763918,245
4,CCOc1ccccc1C(=O)Cc1ccccc1C(=O)O,0.723423,0.425000,1.702171,245
5,COC(=O)Cc1ccccc1C(=O)O,0.698825,0.416667,1.677179,163
6,CC(=O)c1ccccc1OCC(=O)Cc1ccccc1C(=O)O,0.680453,0.439024,1.549920,245
7,O=C(COc1ccccc1C(=O)O)Nc1ccccc1C(=O)O,0.653315,0.435897,1.498782,163
8,COc1cccc(OC)c1C(=O)Cc1ccccc1C(=O)O,0.634823,0.425000,1.493701,245
9,Cc1cc(NC(=O)COc2ccccc2C(=O)O)c(C(=O)O)cc1C,0.590103,0.409091,1.442473,245
